# AI Literacy Lab: Model Training and Fine-tuning

This notebook contains the coding exercises for the lab.  
Each task has its own section below - **only work on the task your instructor tells you to.**

| Section | What it covers |
|---------|---------------|
| **Task 2** | Loading and testing GPT-2 |
| **Task 4** | Experimenting with different sampling methods |
| **Task 5** | Fine-tuning GPT-2 on new data |

---
## Task 2: Loading GPT-2

In this section we load GPT-2, a publicly available language model with 124 million parameters.  
Run the cells below **in order**. The first cell may take a few minutes as it downloads the model.

---
### Before You Start

Make sure you have watched the short video on the tutorial website that explains how to open this notebook in Google Colab and how to enable the **free GPU**. You will need the GPU to run the code efficiently.

**How to run a cell:** Click the **play button** (▶) to the left of any code cell, or press **Cmd + Enter** on Mac, or **Ctrl + Enter** on Windows/Linux.


### Setup

The cell below installs the required Python libraries. **If you are using Google Colab, you can skip this cell** as these libraries are already pre-installed. If you are running this notebook elsewhere, run it once before proceeding.


In [ ]:
# Only run this if you are NOT using Google Colab
!pip install -q transformers datasets accelerate

Run the cell below to import the libraries and set up the environment. This may take a minute or two.


In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("Tip: Go to Runtime > Change runtime type > GPU to speed things up!")

# Load the GPT-2 model and tokenizer
print("Loading GPT-2 model... this may take a few minutes.")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

# GPT-2 needs a pad token setting
tokenizer.pad_token = tokenizer.eos_token

print("GPT-2 loaded successfully!")
print(f"Model size: {sum(p.numel() for p in model.parameters()):,} parameters")

# This helper function generates text from a prompt.
# You do NOT need to modify this cell - just run it once.

def generate_text(prompt, max_length=100, **kwargs):
    """Generate text from a prompt using the current model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_length,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("Generation function ready!")

In [ ]:
# Let's test GPT-2 with a simple prompt!
# Try changing the text in quotes to see different outputs

prompt = "Once upon a time"

output = generate_text(prompt, max_length=80)
print(output)

Don't move on to task 4 yet! Instead, return to the website and move to task 3.

---
## Task 4: Token Prediction - Sampling Methods

When GPT-2 predicts the next token, it produces a list of probabilities for every possible word.  
But how does it **choose** which one to actually output? That depends on the **sampling method**.

The cell below runs the **same prompt** through four different sampling methods so you can compare the outputs side by side. Each method uses a different strategy to pick the next token from the probability list. The samples we will try are:

- **Greedy**
- **5-Beam Search**
- **Contrastive Search**
- **Random Sampling**

**What to change:**
- Edit the `prompt` (the text in quotes) to try different inputs
- Edit `max_length` to generate more or fewer tokens
- Run the cell and compare how the four methods differ

**For discussion question 1**, your group should try **5 different prompts** and record the outputs in a table. You can re-run the cell each time you change the prompt.


In [ ]:
# Change the prompt and max_length below, then run this cell.
# All four methods will use the same prompt so you can compare them.

prompt = "I like playing Minecraft"     # <-- change this
max_length = 100                        # <-- change this

# --- Method 1: Greedy ---
# Picks the single most likely token at every step.
output = generate_text(prompt, max_length, do_sample=False)
print("--- Greedy Output ---")
print(output)

print()

# --- Method 2: 5-Beam Search ---
# Explores multiple sequences in parallel and picks the best overall.
output = generate_text(prompt, max_length, num_beams=5, no_repeat_ngram_size=2, early_stopping=True)
print("--- 5-Beam Output ---")
print(output)

print()

# --- Method 3: Contrastive Search ---
# Balances likelihood with diversity to reduce repetition.
output = generate_text(prompt, max_length, penalty_alpha=0.6, top_k=4, custom_generate="transformers-community/contrastive-search")
print("--- Contrastive Search Output ---")
print(output)

print()

# --- Method 4: Random Sampling ---
# Randomly picks from the top likely tokens, adding variety.
output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)
print("--- Random Sampling Output ---")
print(output)


### Done with Task 4?

Head back to the tutorial website to complete **discussion questions 2 and 3** with your group.


---
## Task 5: Fine-Tuning GPT-2

In this section we **fine-tune** GPT-2 - continuing its training on a small, focused dataset  
so that its outputs shift toward a particular style or topic.

### Stage 1: Fine-tune on a cooking recipe dataset

The code below fine-tunes GPT-2 on a small collection of cooking-related sentences.  
After fine-tuning, the model should be more likely to generate recipe-style text.

**Just run the cells below in order** - you do not need to change anything for Stage 1.

In [ ]:
# Import the libraries we need for fine-tuning
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

# This is our training data - a list of cooking-related sentences.
# The model will learn from the patterns in these sentences.
recipe_texts = [
    "Preheat the oven to 180 degrees Celsius and line a baking tray with parchment paper.",
    "Mix the flour, sugar, and baking powder together in a large bowl until well combined.",
    "Melt the butter in a saucepan over low heat, then stir in the chocolate chips until smooth.",
    "Dice the onions and garlic finely, then sauté in olive oil until translucent.",
    "Season the chicken with salt, pepper, and paprika before placing it in the oven.",
    "Bring a large pot of salted water to a boil and cook the pasta for 8 to 10 minutes.",
    "Whisk the eggs, milk, and vanilla extract together until the mixture is light and fluffy.",
    "Spread the tomato sauce evenly over the pizza dough, then add your favourite toppings.",
    "Simmer the soup on low heat for 30 minutes, stirring occasionally to prevent sticking.",
    "Fold the whipped cream gently into the chocolate mixture to keep the batter airy.",
    "Grill the vegetables on high heat for 5 minutes on each side until slightly charred.",
    "Roll out the pastry dough on a floured surface until it is about 3 millimetres thick.",
    "Toast the bread slices until golden brown, then spread avocado and a pinch of chilli flakes.",
    "Marinate the salmon in soy sauce, ginger, and honey for at least 20 minutes before cooking.",
    "Combine the rice, coconut milk, and sugar in a pot and cook on medium heat until thickened.",
    "Chop the fresh herbs including basil, parsley, and coriander to sprinkle over the finished dish.",
    "Beat the butter and sugar together until pale and creamy before adding the eggs one at a time.",
    "Layer the lasagne sheets with bolognese sauce and bechamel, finishing with a layer of cheese.",
    "Knead the bread dough for 10 minutes until it becomes smooth and elastic to the touch.",
    "Roast the sweet potatoes at 200 degrees for 25 minutes until soft and caramelised on the edges.",
    "Stir the risotto continuously while adding warm stock one ladle at a time.",
    "Blend the bananas, strawberries, yoghurt, and ice together to make a smooth fruit smoothie.",
    "Coat the fish fillets in seasoned flour, then pan-fry in butter until crispy and golden.",
    "Let the cake cool completely in the tin before turning it out and adding the icing.",
    "Toss the salad leaves with olive oil, lemon juice, salt, and freshly ground black pepper.",
]

# Convert our list of sentences into a Dataset object that the trainer can use
dataset = Dataset.from_dict({"text": recipe_texts})

# Tokenise the text - this converts each sentence into numbers (token IDs)
# that the model can understand, just like we saw in Task 1
def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Dataset prepared: {len(recipe_texts)} recipe examples")

In [ ]:
# Set up the training configuration
training_args = TrainingArguments(
    output_dir="./fine_tuned_recipes",  # Where to save the model
    num_train_epochs=5,                  # How many times to loop through the data
    per_device_train_batch_size=2,       # How many examples to process at once
    learning_rate=5e-5,                  # How much to adjust the model each step
    save_strategy="no",                  # Don't save checkpoints (saves disk space)
    logging_steps=5,                     # Print progress every 5 steps
    report_to="none",                    # Don't send logs to external services
)

# The data collator handles formatting the data for language modelling
# mlm=False means we are doing causal (next-token) language modelling, not masked
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# The Trainer brings everything together: the model, the data, and the settings
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Start fine-tuning! This continues the model's training on our recipe data
print("Fine-tuning started...")
trainer.train()
print("\nFine-tuning complete!")

### Compare: Before vs After Fine-Tuning

The cell below generates text using the **fine-tuned** model, then loads a **fresh pre-trained** copy of GPT-2 and generates text with the same prompt. This lets you see the difference side by side.

For **discussion question 2**, change the prompt and run this cell **5 times** with different prompts. Record the outputs in a table.


In [ ]:
# Change the prompt below, then run this cell to compare before vs after.

prompt = "Hi there"              # <-- change this
max_length = 100                        # <-- change this

# Generate with the FINE-TUNED model (the one we just trained)
finetuned_output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)

# Temporarily load a fresh PRE-TRAINED GPT-2 for comparison
import copy
finetuned_model = model  # save reference to fine-tuned model
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
pretrained_output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)
model = finetuned_model  # restore the fine-tuned model

print("--- Pre-trained GPT-2 ---")
print(pretrained_output)
print()
print("--- Fine-tuned GPT-2 (recipes) ---")
print(finetuned_output)


---
### Stage 2: Fine-tune on your own data

Now it is your turn! Write your own training sentences in the cell below.

**Important tips:**
- Your sentences must share something in common - a **topic**, a **style**, or a **pattern** - so the model can learn from them
- For example: all about sports, all written like a pirate, all about your university, all in the style of a news headline
- **Come up with the sentences as a group** - aim for at least **20-30 sentences** together - the more you provide, the better the model will learn
- Each sentence should be a complete thought, roughly 1-2 lines long

In [ ]:
# We need to reload a fresh copy of GPT-2 so we start from scratch.
# Stage 1 changed the model's weights - we don't want those recipe
# patterns interfering with our new fine-tuning.
print("Reloading a fresh GPT-2 model...")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
print("Fresh model loaded!")

In [ ]:
# WRITE YOUR OWN TRAINING SENTENCES BELOW
# As a group, come up with sentences and replace the examples with your own!
# Remember: all sentences should share a common theme or style.
# The model will learn from the patterns these sentences have in common.

my_data = [
    "Write your first sentence here.",
    "Write your second sentence here.",
    "Write your third sentence here.",
    "Write your fourth sentence here.",
    "Write your fifth sentence here.",
    # Keep adding more sentences below!
    # Aim for at least 20-30 sentences.
]

print(f"You have written {len(my_data)} sentences.")
if len(my_data) < 20:
    print("Try to add more sentences for better results! Aim for at least 20.")
else:
    print("Good amount of data!")

In [ ]:
# Prepare your data the same way we did in Stage 1:
# convert to a Dataset, then tokenise it
custom_dataset = Dataset.from_dict({"text": my_data})
custom_tokenized = custom_dataset.map(tokenize, batched=True, remove_columns=["text"])

# Same training settings as Stage 1
training_args = TrainingArguments(
    output_dir="./fine_tuned_custom",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    save_strategy="no",
    logging_steps=5,
    report_to="none",
)

# Create the trainer and start fine-tuning on your data
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=custom_tokenized,
    data_collator=data_collator,
)

print("Fine-tuning on your data...")
trainer.train()
print("\nFine-tuning complete!")

In [ ]:
# Test your custom fine-tuned model!
# Change the prompt to something related to your group's theme.
# For example, if your sentences were about sports, try a sports-related prompt.
# Run this cell multiple times with different prompts to see how well
# the model picked up on your group's data.

prompt = "Prompt here"                             # <-- change this
max_length = 100                                   # <-- change this

output = generate_text(prompt, max_length, do_sample=True, top_k=50, temperature=0.7)
print("--- Your Fine-tuned GPT-2 ---")
print(output)

---
## All Done!

Once you have completely finished with this notebook, **disconnect your runtime** to free up the GPU for others:

**Runtime > Disconnect and delete runtime**

Then head back to the tutorial website to complete any remaining discussion questions with your group.
